# Task 2
- Group: 36
- Candidate number: 4, 49, 63, 82

Troubleshooting the 4-sided dice code

The original code consists of a variety of errors:

    - Syntax errors
    - Runtime errors
    - Logical errors

Below we have identified and explained all the bugs, followed up with two versions of the fixed code:

    1. fixed code with minimal fixes
    2. fixed code + refactoring


### Bugs we found

Going through the code we ended up identifying 12 separate issues. Some are
straightforward syntax errors that prevent the code from running at all,
others are sneakier. The code runs fine but the game behaves incorrectly.
We've also flagged a couple of things that aren't strictly bugs, but design
choices that make the code harder to read and maintain.

| # | Where | Type | What's wrong | What we did |
|---|-------|------|--------------|-------------|
| 1 | `random.randint(...)` after `import random as rnd` | Runtime | The module was imported under the alias `rnd`, so referring to it as `random` throws a `NameError` | Switched to `rnd.randint(...)` |
| 2 | `randint(1, 5)` | Logical | A 4 sided dice should give values from 1 to 4, not 1 to 5 | Changed the upper bound to 4 |
| 3 | `('1 ', '2 ', '3 ', '4 ')` | Logical | Each option has a trailing space, so `input()` (which doesn't include one) will never match. The loop never exits | Removed the spaces |
| 4 | `guess = input()` inside `get_dice_guess()` | Logical | The loop checks `guesss` (three s's) but the new input gets stored in `guess`, so the loop variable never updates and the function spins forever | Picked one name and used it consistently |
| 5 | `if dice = guess:` (three places) | Syntax | `=` assigns a value, it doesn't compare. Python raises a `SyntaxError` here | Changed all three to `==` |
| 6 | The last `if dice = guess` | Syntax | Missing the colon at the end of the line | Added the `:` |
| 7 | `dice == guess` after `input()` | Logical | `dice` is an integer (from `randint`), but `guess` is a string (from `input`). The comparison silently always returns `False` | Wrapped the input in `int()` before comparing |
| 8 | `get_dice_guess()` is defined but never called | Logical | The function exists but nothing ever invokes it, so the user never gets prompted | Actually called it where input was needed |
| 9 | `roll_dice()` is also never called | Logical | The whole game logic is wrapped in a function that nothing runs | Added a call at the bottom of the script |
| 10 | Mix of `guess` and `guesss` throughout | Logical | These are two different variables as far as Python is concerned, which breaks the intended flow | Standardized on a single name |
| 11 | Four levels of nested `if/else` for the 4 attempts | Design choice | It works, but it's repetitive and hard to extend (what if we wanted 5 attempts?) | Rewrote with a `for` loop in the refactored version |
| 12 | Globals `dice = 0` and `guess = 0` at the top | Design choice | Functions create their own local versions, so the globals just sit there doing nothing | Removed them |

## Version 1: Fixed code with minimal fix:

In [9]:
import random as rnd

def get_dice_guess(): # Changed name to snake casing
    guess = 0
    while guess not in ('1', '2', '3', '4'): # Removed space in '1 '
        print("Guess the number on a 4-sided dice! Enter a number between 1 and 4:")
        guess = input("Enter a number between 1 and 4:")
        print("Your guess: ", guess)
    return int(guess) # removed triple s and moved out of while loop

def roll_dice():
    dice = rnd.randint(1, 4) # changed name to rnd as it is alias for random, and changed from 5 to 4 since it is four sides only
    return dice # returning a number to set global dice to

dice = roll_dice()
guess = get_dice_guess()

if dice == guess:
    print("You are good!")
else:
    print("Sorry! Try to guess again!")
    guess = get_dice_guess() # Calls get_dice_guess to verify userinput
    if dice == guess: # added double == to compare value, instead of single = to set value
        print("Congrats! You got it!")
    else:
        print("Sorry! Your third chance!")
        guess = get_dice_guess()
        if dice == guess:
            print("Congrats! You got it!")
        else:
            print("Sorry! You still can try!")
            guess = get_dice_guess()
            if dice == guess: # Moved in to else
                print("Congrats! You got it!")
            else:
                print("Nope. You are really bad at this game.")

print("Dice: ", dice)

Guess the number on a 4-sided dice! Enter a number between 1 and 4:
Your guess:  1
Sorry! Try to guess again!
Guess the number on a 4-sided dice! Enter a number between 1 and 4:
Your guess:  2
Congrats! You got it!
Dice:  2


## Version 2: Fixed code + refactoring:

In [10]:
import random as rnd

def get_dice_guess():
    # Collects a guess from the user (numbers 1–4).
    while True:
        guess_input = input('Guess the number on a 4-sided dice! Enter a number between 1 and 4: ')
        if guess_input in ('1', '2', '3', '4'):
            return int(guess_input)
        print('Invalid input – try again!') #error-handler for invalid input

def roll_dice():
    # Main logic of the game: 4 tries to guess the correct dice number.
    dice = rnd.randint(1, 4)

    for attempt in range(4): #For loop that iterates four times since the user only has 4 tries.
        guess = get_dice_guess()
        if guess == dice:
            print(f'Bravo! You guessed the {guess} correct number! Total attempts: {attempt + 1}.')
            return
        elif attempt < 3:
            print(f'{guess} is the wrong number! Try again! Attempt: {attempt + 1}.')

    print(f'Nope. You are really bad at this game :) The correct number was: {dice}.')

roll_dice()

1 is the wrong number! Try again! Attempt: 1.
2 is the wrong number! Try again! Attempt: 2.
3 is the wrong number! Try again! Attempt: 3.
Bravo! You guessed the 4 correct number! Total attempts: 4.


## What we've changed:

- Implemented an endless while loop where the only way to exit the loop by returning the guess_input
- Added error handler if the input was invalid example: 5
- Changed from nested if-else blocks to a for loop to avoid repetitive code.
- implemented variables in the print statement to indicate user's guess and total attempts.
- removed unnecessary global variables

### Reflection: which version is better?

Both versions solve the original problem, but they make different trade-offs.

Version 1 keeps the original structure intact and just fixes the bugs. It's
useful for understanding what the original developer intended, and it's the
safest minimal change if the surrounding code depends on this structure.

Version 2 rewrites the game loop with a `for` loop and a separate input
function with built-in validation. It's shorter, easier to extend (changing
the number of attempts is a one-line edit), and harder to introduce new
bugs into. If we owned this code and had to maintain it long-term, we
would go with Version 2.

The exercise shows how the same problem can have very different solutions
depending on whether you're patching existing code or designing it from
scratch.